[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/motor_loop/motor_loop.ipynb)

# Loop Subdivision of Motors

This example computes Loop subdivision of a triangle mesh whose vertices carry rigid placements of frames instead of points. A Loop step refines a mesh by averaging: every vertex of the finer mesh is a fixed average of vertices of the coarser one, with shares that depend only on how the mesh is connected. A ribbon wound up by a twist shows what averaging placements buys over averaging positions. Refined as points it wrings itself thin, because the average of two positions cuts across the turn between them; refined as placements it keeps its width, because the average of two frames is the frame turned halfway, out on the arc between them.

Each placement is a motor, and one Loop step is a linear map from the coarser mesh's vertex sites to the finer one's that applies to whatever sits there, `step * motors`. A weighted sum of motors is not a motor, so every averaged motor is normalized back after each step, `(step * motors).normalized()`: average in the algebra, return to the group. The surface is the origin each motor carries, `motors >> origin`.

In the matrix and dual-quaternion notation of subdivision surfaces, a Loop step reads as a subdivision matrix $S$ acting on the stacked vertex positions $P' = S P$, while motor subdivision applies the same $S$ to the stacked unit dual quaternions and normalizes each row back to unit norm.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, stack
from numga.algebras import PGA3D as ga
from examples import instantiate
from examples.util.animation import show_animation
from examples.surfaces.motor_loop import render
from examples.surfaces.motor_loop.core import grid_faces

mv = NumpyContext(ga).multivector
Line = ga.gatype.bivector()                         # Bivector+
Point = ga.gatype.antivector()                      # Antivector-
Motor = ga.gatype.rotor()                           # Rotor+
Mesh = instantiate("examples.util.mesh", ga).Mesh


## 1. Subdividing motors and points on a mesh

A Loop step is fixed by the faces alone: a linear map from the vertex sites of the coarser mesh to those of the finer one, here with the pinned corners keeping themselves. The map does not care what sits at the sites. Applied to motors it averages them, and every averaged motor is normalized back to a motor after each step. Applied to the points the motors carry, nothing happens between the steps, so the steps compose into one map. The two surfaces are drawn side by side, the points on the left and the motors on the right; a frame is drawn as its origin and the ends of its three axes.

In [ ]:
# The ribbon's control mesh: vertices along z and across x, and its length and width. The number of Loop
# subdivision steps and the length of the rendered frame axes.
n_along, n_across = 5, 3
length, width = 4.0, 1.2
n_loop_steps = 3
axis_length = 0.3

In [ ]:
# The ribbon's vertices on a grid along z and across x, with its four corners pinned.
z_coords, x_coords = (
    grid.reshape(-1)
    for grid in np.meshgrid(
        np.linspace(-length / 2, length / 2, n_along),
        np.linspace(-width / 2, width / 2, n_across),
        indexing="ij",
    )
)
corners = [0, n_across - 1, n_along * n_across - n_across, n_along * n_across - 1]
pinned = np.isin(np.arange(n_along * n_across), corners)
control_faces = grid_faces(n_along, n_across)

# The refined mesh holds the Loop steps from the control mesh.
refined_mesh = Mesh(n_along * n_across, control_faces).loop_subdivided(pinned, n_loop_steps)
# Points need nothing between the steps, so they take the steps composed into one map.
point_subdivision = refined_mesh.subdivision                                   # Scalar+[refined <- vertices]


def subdivide_motors(motors: Motor) -> Motor:
    """Motor subdivision averages the motors at every step and normalizes each average back to a motor."""
    for step in refined_mesh.steps:
        motors = (step * motors).normalized()                                  # [...] Motor+[finer]
    return motors


markers = stack([
    mv.w.dual(),
    mv.point([axis_length, 0.0, 0.0]),
    mv.point([0.0, axis_length, 0.0]),
    mv.point([0.0, 0.0, axis_length]),
])                                                                             # [axes + 1] Point-
origin = markers[0]                                                            # [] Point-

## 2. Winding up the ribbon

One end of the ribbon is held fixed while the far end is wound one and a half turns and released, with the twist angle growing toward the far end as the square of the distance along the ribbon. Point subdivision on the left averages vertex positions across the turn, pulling the middle of the ribbon inward toward the twist axis like a wrung strand. Motor subdivision on the right averages the vertex motors and normalizes them after each step, placing every subdivided frame partway around the turn at its full distance from the axis. The control frames are drawn with red, green, and blue $x$, $y$, and $z$ axes over the dashed control net.

In dual-quaternion notation a motor reads as a unit dual quaternion and each motor subdivision step reads as dual-quaternion linear blending followed by normalization, while point subdivision reads as linear blend skinning with its characteristic candy-wrapper collapse.

In [ ]:
# The twist angle of the far end at the peak of the wind-up, the frame count, and the frame duration.
twist_angle = 3 * np.pi
n_frames = 48
duration_ms = 60

In [ ]:
# Compute the wind-up frames; the animation plays them forward and back.
twist_angles = twist_angle * (1 - np.cos(np.linspace(0, np.pi, n_frames // 2 + 1))) / 2  # [frames]
twist_profile = ((z_coords + length / 2) / length) ** 2                        # [vertices]
# Place each frame on the flat ribbon along z and across x, then twist around the z axis.
rest_motors = (mv.zw * (z_coords / 2)).exp() * (mv.xw * (x_coords / 2)).exp()  # [vertices] Motor+
twist_motors = (mv.xy * (-twist_angles[:, None] * twist_profile / 2)).exp()    # [frames, vertices] Motor+
winding_motors = (twist_motors * rest_motors).field()                          # [frames] Motor+[vertices]
control_frames = winding_motors[:, None] >> markers                            # [frames, axes + 1] Point-[vertices]
point_surfaces = point_subdivision * (winding_motors >> origin)                # [frames] Point-[refined]
motor_surfaces = subdivide_motors(winding_motors) >> origin                    # [frames] Point-[refined]
surfaces = stack([point_surfaces, motor_surfaces], axis=1)                     # [frames, schemes] Point-[refined]

show_animation(render.animate(control_frames, surfaces, control_faces, refined_mesh.faces), "motor_loop_ribbon", duration_ms,
               forward_then_back=True)

## 3. Turning the frames in place

When the control points stay on the flat ribbon while every vertex frame turns back and forth about a line through its own origin at its own phase, no control point moves and the point-subdivided surface on the left stays flat. The motor-subdivided surface on the right bends: the normalized average of two differently oriented frames follows the screw motion between them, so relative rotation between neighbouring frames bows the subdivided surface out of the plane. A rigid rotation applied in common to every control frame about its own origin changes nothing, because right-multiplying all vertex motors by the same rotor cancels when the subdivided motor acts on the origin; only relative turns between neighbouring frames bend the surface.

In [ ]:
# The amplitude and random seed for turning each vertex frame in place about its own axis.
turn_amplitude = np.pi / 3
turn_seed = 2

In [ ]:
rng = np.random.default_rng(turn_seed)
axes = mv(Line.derive.nondegenerate, rng.normal(size=(len(z_coords), 3))).normalized()  # [vertices] Line+
phases = rng.uniform(0, 2 * np.pi, len(z_coords))
angles = turn_amplitude * np.sin(np.linspace(0, 2 * np.pi, n_frames, endpoint=False)[:, None] + phases)  # [frames, vertices]
local_turns = (axes * (-angles / 2)).exp()                                     # [frames, vertices] Motor+
turning_motors = (rest_motors * local_turns).field()                           # [frames] Motor+[vertices]
turning_frames = turning_motors[:, None] >> markers                            # [frames, axes + 1] Point-[vertices]
turning_point_surfaces = point_subdivision * (turning_motors >> origin)        # [frames] Point-[refined]
turning_motor_surfaces = subdivide_motors(turning_motors) >> origin            # [frames] Point-[refined]
turning_surfaces = stack([turning_point_surfaces, turning_motor_surfaces], axis=1)  # [frames, schemes] Point-[refined]

show_animation(render.animate(turning_frames, turning_surfaces, control_faces, refined_mesh.faces), "motor_loop_turning", duration_ms)

In [ ]:
# checks
from examples.surfaces.motor_loop import core

np.testing.assert_allclose((subdivide_motors(winding_motors) - core.subdivide_motors(refined_mesh, winding_motors)).kernel, 0.0, atol=1e-12)

# Untwisted, the two schemes agree; wound up, point subdivision wrings the ribbon toward its axis
# while motor subdivision preserves its width. Joining a unit point with the z axis line forms a
# plane whose norm is the point's radial distance from the axis.
centre_line = origin & mv.point([0.0, 0.0, 1.0])                               # [] Line-
distance = (centre_line & surfaces[[0, n_frames // 2]].batch()).norm().mean(axis=-1)  # [cases, schemes] Scalar+
(points_rest, motors_rest), (points_peak, motors_peak) = distance
assert (points_rest - motors_rest).abs() < 1e-12
assert points_peak < 0.7 * points_rest and motors_peak >= motors_rest

# Turning the frames in place leaves the control points fixed while bending the motor surface.
point_batch, motor_batch = turning_point_surfaces.batch(), turning_motor_surfaces.batch()  # [frames, refined] Point-, Point-
assert np.all((point_batch & point_batch[:1]).norm() < 1e-12)
assert np.any((motor_batch & point_batch).norm() > 0.05)

# A rigid turn shared by every control frame leaves the subdivided surface unchanged.
shared_turn = (mv.yz * (-0.7 / 2)).exp()                                       # [] Motor+
shared_motors = stack([rest_motors, rest_motors * shared_turn]).field()     # [cases] Motor+[vertices]
cases = (subdivide_motors(shared_motors) >> origin).batch()                   # [cases, refined] Point-
assert np.all((cases[0] & cases[1]).norm() < 1e-12)